In [ ]:
import numpy as np
import pyscf
import py3Dmol
from uncertainties import ufloat
import matplotlib.pyplot as plt


In [ ]:
from oneqmc.analysis.visual import show_mol
from oneqmc.convert_geo import load_molecules
from oneqmc.analysis.plot import set_defaults

set_defaults()

In [ ]:
params = np.load(
    "../../experiment_results/09_azulene_naphthalene/finetune_params.npz"
)

In [ ]:
naphthalene = load_molecules(
    "../../../data/naphthalene-azulene/naphthalene"
)[0]
azulene = load_molecules(
    "../../../data/naphthalene-azulene/azulene"
)[0]

#### Isosurfaces of envelopes

In [ ]:
class CubeDataFormatter:
    def __init__(self, mol, nx=50, ny=50, nz=50):
        self.charges = mol.charges
        self.coords = mol.coords
        margin = 3.0
        extent = np.max(mol.coords, axis=0) - np.min(mol.coords, axis=0) + 2 * margin
        self.box = np.diag(extent)
        self.boxorig = np.min(mol.coords, axis=0) - margin

        self.nx = nx
        self.ny = ny
        self.nz = nz
        self.xs = np.linspace(0, 1, nx)
        self.ys = np.linspace(0, 1, ny)
        self.zs = np.linspace(0, 1, nz)

    def get_coords(self):
        frac_coords = np.stack(np.meshgrid(self.xs, self.ys, self.zs), axis=-1)
        # permuting x<->y is necessary to match weird ordering of cube format
        return np.einsum("yxzi,ij->xyzj", frac_coords, self.box) + self.boxorig

    def __call__(self, field) -> str:
        assert field.ndim == 3
        assert field.shape == (self.nx, self.ny, self.nz)
        comment = ""
        string = ""
        string += comment + "\n"
        string += "Created by OneQMC CubeFormatter\n"
        string += f"{len(self.coords):5d}"
        string += "{:12.6f}{:12.6f}{:12.6f}\n".format(*tuple((self.boxorig).tolist()))
        dx = self.xs[-1] if len(self.xs) == 1 else self.xs[1]
        dy = self.ys[-1] if len(self.ys) == 1 else self.ys[1]
        dz = self.zs[-1] if len(self.zs) == 1 else self.zs[1]
        delta = (self.box.T * np.stack([dx, dy, dz])).T
        string += (
            f"{self.nx:5d}{delta[0,0]:12.6f}{delta[0,1]:12.6f}{delta[0,2]:12.6f}\n"
        )
        string += (
            f"{self.ny:5d}{delta[1,0]:12.6f}{delta[1,1]:12.6f}{delta[1,2]:12.6f}\n"
        )
        string += (
            f"{self.nz:5d}{delta[2,0]:12.6f}{delta[2,1]:12.6f}{delta[2,2]:12.6f}\n"
        )
        for charge, coord in zip(self.charges, self.coords):
            string += "%5d%12.6f" % (charge, 0.0)
            string += "{:12.6f}{:12.6f}{:12.6f}\n".format(*tuple((coord).tolist()))

        # Sync to CPU if not there already
        field = np.asarray(field)
        for ix in range(self.nx):
            for iy in range(self.ny):
                for iz0, iz1 in pyscf.lib.prange(0, self.nz, 6):
                    fmt = "%13.5E" * (iz1 - iz0) + "\n"
                    string += fmt % tuple(field[ix, iy, iz0:iz1].tolist())

        return string


def show_isosurface(
    field_data,
    iso_value: float = 0.05,
    view=None,
):
    if view is None:
        view = py3Dmol.view()
    view.addVolumetricData(
        field_data,
        "cube",
        {
            "isoval": -iso_value,
            "smoothness": 5,
            "opacity": 0.8,
            "volformat": "cube",
            "color": "blue",
        },
    )
    view.addVolumetricData(
        field_data,
        "cube",
        {
            "isoval": iso_value,
            "smoothness": 5,
            "opacity": 0.8,
            "volformat": "cube",
            "color": "red",
        },
    )
    return view

In [ ]:
formatter = CubeDataFormatter(naphthalene, nx=100, nz=100)

In [ ]:
grid = formatter.get_coords()

In [ ]:
def envelope_fn(x, exponents, centers, coefs):
    r = np.linalg.norm(centers - x[..., None, :], axis=-1)
    exps = np.exp(-exponents * r[..., None])
    return np.einsum("...ij,ij->...", exps, coefs)

In [ ]:
orb_idx = 34
coef = params["naphthalene_2_se_envelope_up_feature_selector"][
    orb_idx, :, :, 0
]
field_values = envelope_fn(
    grid,
    params["naphthalene_2_exponents"].squeeze(0),
    naphthalene.coords,
    coef,
)
# Easier to visualize if it integrates to 1
field_values /= np.abs(field_values).sum()
field_data = formatter(field_values)
view = show_mol(naphthalene)
show_isosurface(field_data, view=view, iso_value=1e-5)

In [ ]:
orb_idx = 17
coef = params["naphthalene_2_se_envelope_up_feature_selector"][
    orb_idx, :, :, 0
]
field_values = envelope_fn(
    grid,
    params["naphthalene_2_exponents"].squeeze(0),
    naphthalene.coords,
    coef,
)
# Easier to visualize if it integrates to 1
field_values /= np.abs(field_values).sum()
field_data = formatter(field_values)
view = show_mol(naphthalene)
show_isosurface(field_data, view=view, iso_value=1e-5)

## Sturcture figures

In [ ]:
view = show_mol(naphthalene)
view.rotate(90, "z")

In [ ]:
view = show_mol(azulene)
view.rotate(36, "z")

## Convergence plot

In [ ]:
%config InlineBackend.figure_format = 'retina'

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image

training_iterations = [48_000, 56_000, 64_000, 72_000, 80_000]
reaction_energies = [ufloat(31.87, 1.2), ufloat(36.6368, 0.7353), ufloat(37.5258, 0.6017), ufloat(36.6801, 0.5550), ufloat(36.9793, 0.9101)]
ccsdpt_cbs_reference = 36.82

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(
    training_iterations,
    [r.n for r in reaction_energies],
    color="#1C909F",
    label="Orbformer",
)
ax.axhline(ccsdpt_cbs_reference, color="k", linestyle="--", label="CCSD(T)/CBS")
ax.fill_between(
    [40_000, 90_000],
    [ccsdpt_cbs_reference - 1.0],
    [ccsdpt_cbs_reference + 1.0],
    alpha=0.2,
    color="gray",
)
ax.set_xlabel("Orbformer finetuning iterations")
ax.set_ylabel("Reaction energy (kcal/mol)")
ax.set_ylim(30.5, 41.0)
ax.set_xlim(44_000, 85_000)
ax.set_xticks([50_000, 60_000, 70_000, 80_000])
ax.set_yticks([32, 34, 36, 38, 40])
ax.legend(loc=(0.02, 0.75))

struct_dir = "../../experiment_results/09_azulene_naphthalene/molecule_images"
naphthalene_img = np.array(Image.open(f"{struct_dir}/naphthalene-structure.png"))
azulene_img = np.array(Image.open(f"{struct_dir}/azulene-structure.png"))

x_pos, y_pos = 76_500, 33.0
gap = 17000
zoom = 0.13
im_naph = OffsetImage(naphthalene_img, zoom=zoom)
ab_naph = AnnotationBbox(im_naph, (x_pos - gap, y_pos), frameon=False)
ax.add_artist(ab_naph)
im_azul = OffsetImage(azulene_img, zoom=zoom)
ab_azul = AnnotationBbox(im_azul, (x_pos, y_pos), frameon=False)
ax.add_artist(ab_azul)

ax.annotate(
    "",
    xy=(x_pos - gap / 2 + 1200, y_pos),
    xytext=(x_pos - gap / 2 - 2200, y_pos),
    arrowprops=dict(arrowstyle="->", lw=1.5, color="k"),
)
fig.savefig("reaction_energy_convergence.pdf", dpi=300, bbox_inches="tight")